# Diagnostic — which benefits are extracting vs failing

Run this against your `compare_payload_vs_output` results. Identifies whether zero-row benefits are caused by data mismatch or LLM failure.

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd

# ── Same paths as the comparison notebook ─────────────────────────────────────
PAYLOAD_PATH = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')
OUTPUT_PATH  = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_output.json')

# Load both
payload_raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
payload_rows = payload_raw['pbp'] if isinstance(payload_raw, dict) and 'pbp' in payload_raw else payload_raw

output_rows = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
if isinstance(output_rows, dict) and 'results' in output_rows:
    output_rows = output_rows['results']

print(f'Payload: {len(payload_rows):,} rows | Output: {len(output_rows):,} rows')

# Mirror BENEFIT_TARGETS from build_benefits.py
BENEFIT_TARGETS = [
    ('600',  'Monthly Premium',            [],             ['Monthly Premium', 'Plan Premium']),
    ('610',  'Health Plan Deductible',     [],             ['Health Plan Deductible', 'Medical Deductible']),
    ('611',  'Drug Deductible',            [],             ['Drug Deductible', 'Rx Deductible', 'Enter Deductible Amount']),
    ('614',  'Health Monthly Premium',     [],             ['Health Monthly Premium', 'Health Premium']),
    ('615',  'Drug Monthly Premium',       [],             ['Drug Monthly Premium', 'Rx Premium', 'Part D Premium']),
    ('616',  'Part B Premium Reduction',   [],             ['Part B Premium', 'Part B Reduction', 'Part B giveback']),
    ('620',  'Out-of-Pocket Spending',     [],             ['MOOP', 'Max Enrollee Cost', 'Out of Pocket']),
    ('700',  'Tier Names',                 [],             ['Rx Tier', 'Formulary Tier', 'Tier Names']),
    ('710',  'Initial Coverage',           [],             ['Initial Coverage Phase', 'Rx Setup']),
    ('711',  'Retail Pharmacy',            [],             ['Retail Pharmacy', 'Initial Coverage Phase']),
    ('730',  'Catastrophic Coverage',      [],             ['Catastrophic Coverage']),
    ('740',  'Formulary Exception',        [],             ['Formulary Exception']),
    ('755',  'Preferred Mail Order',       [],             ['Preferred Mail Order']),
    ('760',  'Standard Mail Order',        [],             ['Standard Mail Order']),
    ('800',  'Inpatient Hospital',         ['1a'],         ['Inpatient Hospital']),
    ('810',  'Inpatient Mental Health',    ['1b'],         ['Inpatient Mental Health', 'Inpatient Psychiatric']),
    ('820',  'Skilled Nursing Facility',   ['2'],          ['Skilled Nursing', 'SNF']),
    ('900',  'PCP',                        ['7a'],         ['Primary Care', 'PCP']),
    ('910',  'Specialist',                 ['7b'],         ['Specialist', 'Specialty Care']),
    ('911',  'Telehealth',                 ['7d'],         ['Telehealth', 'Telemedicine', 'Virtual']),
    ('920',  'Chiropractic',               ['8'],          ['Chiropractic']),
    ('930',  'Podiatry',                   ['9a'],         ['Podiatry']),
    ('940',  'Outpatient Mental Health',   ['4a'],         ['Outpatient Mental Health']),
    ('950',  'Outpatient Substance Abuse', ['4b'],         ['Substance Abuse']),
    ('960',  'Outpatient Surgery',         ['4c'],         ['Outpatient Surgery', 'Outpatient Services', 'Ambulatory Surgical']),
    ('970',  'Ambulance',                  ['5'],          ['Ambulance']),
    ('981',  'Emergency',                  ['6a'],         ['Emergency']),
    ('982',  'Urgent Care',                ['6b'],         ['Urgent Care', 'Urgently Needed']),
    ('990',  'Outpatient Rehab',           ['10'],         ['Outpatient Rehabilitation', 'Physical Therapy', 'Occupational Therapy']),
    ('1000', 'DME',                        ['11a'],        ['Durable Medical Equipment', 'DME']),
    ('1020', 'Diabetes',                   ['11c'],        ['Diabetic', 'Diabetes']),
    ('1030', 'Diagnostic/Lab/Radiology',   ['3'],          ['Diagnostic', 'X-Ray', 'Lab', 'Radiology']),
    ('1050', 'Fitness',                    ['13b'],        ['Fitness', 'SilverSneakers']),
    ('1060', 'Meals',                      ['13c'],        ['Meals']),
    ('1200', 'Kidney',                     ['12'],         ['Kidney', 'Renal', 'Dialysis']),
    ('1300', 'Preventive Dental',          ['16a'],        ['Preventive Dental']),
    ('1301', 'Comprehensive Dental',       ['16b'],        ['Comprehensive Dental']),
    ('1400', 'Hearing',                    ['18a', '18b'], ['Hearing Exam', 'Hearing Aid']),
    ('1500', 'Vision',                     ['17a', '17b'], ['Vision', 'Eyewear', 'Eyeglasses']),
    ('1610', 'Prosthetics',                ['11b'],        ['Prosthetic']),
    ('1700', 'Preventive',                 ['14a', '14b', '14c'], ['Preventive', 'Wellness Visit']),
    ('1800', 'Transportation',             ['15'],         ['Transportation']),
    ('1900', 'Acupuncture',                ['13a'],        ['Acupuncture']),
    ('2100', 'OTC',                        ['13e'],        ['Over-the-Counter', 'OTC']),
    ('2110', 'Plan Rating',                [],             ['Plan Rating', 'Star Rating']),
    ('2111', 'Health Plan Rating',         [],             ['Health Plan Rating']),
    ('2112', 'Drug Plan Rating',           [],             ['Drug Plan Rating']),
]
print(f'Benefit targets: {len(BENEFIT_TARGETS)}')


## 1. Output: rows per benefit ID, across all plans

In [ ]:
out_df = pd.DataFrame(output_rows)
out_df['benefitid'] = out_df['benefitid'].astype(str)

bid_counts = out_df.groupby('benefitid').size()
plans_with_bid = out_df.groupby('benefitid')['planid'].nunique()

expected_ids = [t[0] for t in BENEFIT_TARGETS]
expected_names = {t[0]: t[1] for t in BENEFIT_TARGETS}
n_plans = out_df['planid'].nunique()

summary = []
for bid in expected_ids:
    total_rows = int(bid_counts.get(bid, 0))
    plans_seen = int(plans_with_bid.get(bid, 0))
    summary.append({
        'benefitid':   bid,
        'name':        expected_names.get(bid, ''),
        'total_rows':  total_rows,
        'plans_w_data': plans_seen,
        'pct_plans':   f'{100*plans_seen/n_plans:.0f}%',
        'status':      'OK' if plans_seen > 0 else 'ZERO',
    })
sum_df = pd.DataFrame(summary).sort_values('total_rows', ascending=False)
display(sum_df)

zero_benefits = sum_df[sum_df['status']=='ZERO']
print(f'\nBenefits with ZERO rows across all {n_plans} plans: {len(zero_benefits)}')


## 2. For each ZERO benefit — does the input even contain matchable rows?

In [ ]:
def row_matches_target(row, section_codes, keywords):
    category = (row.get('category') or '').lower()
    for code in section_codes:
        if f'({code})' in category or f'({code}1)' in category or f'({code}2)' in category:
            return True
    for kw in keywords:
        if kw.lower() in category:
            return True
    return False

print(f'Diagnosing {len(zero_benefits)} zero-output benefits:\n')
print(f'{"bid":>5} {"name":<28} {"matched":>8} {"distinct_cats":>13}  diagnosis')
print('-' * 100)

# Track totals for headline
n_input_none = 0
n_llm_failed = 0
zero_diag = []

for bid in zero_benefits['benefitid']:
    target = next(t for t in BENEFIT_TARGETS if t[0] == bid)
    _, name, codes, keywords = target

    matched_rows = [r for r in payload_rows if row_matches_target(r, codes, keywords)]
    distinct_cats = sorted({r.get('category','') for r in matched_rows})

    if not matched_rows:
        diag = 'INPUT_HAS_NONE'
        n_input_none += 1
    else:
        diag = f'LLM_FAILED (input has {len(matched_rows)} rows)'
        n_llm_failed += 1

    zero_diag.append({'bid': bid, 'name': name, 'diagnosis': diag})
    print(f'{bid:>5} {name:<28} {len(matched_rows):>8} {len(distinct_cats):>13}  {diag}')
    for c in distinct_cats[:2]:
        print(f'        sample: {c[:90]}')


## 3. What section codes does the input actually use?

Fixed regex — now extracts codes like `7a`, `13a`, `18b1` from anywhere in category strings.

In [ ]:
# FIXED: explicit capture group so m.group(1) works
# Matches things like (7a), (13a), (18b1), 7a, 18b2 — anywhere in the string
section_pattern = re.compile(r'\\(?(\\d+[a-z]\\d?)\\)?')

cat_counter = Counter()
section_counter = Counter()

for r in payload_rows:
    cat = (r.get('category') or '').strip()
    if cat:
        cat_counter[cat] += 1
        for m in section_pattern.finditer(cat):
            section_counter[m.group(1)] += 1

print(f'Distinct categories in payload: {len(cat_counter):,}')
print(f'Distinct section codes found:   {len(section_counter)}\n')

print('Most common section codes:')
print(f'{"code":>8}  rows     example category')
print('-' * 100)
for code, n in section_counter.most_common(40):
    # Find an example category that contains this code
    example = next((c for c in cat_counter if code in c.lower()), '')
    print(f'{"("+code+")":>8}  {n:>6,}  {example[:80]}')


## 4. Top 50 unique categories in the payload (by frequency)

In [ ]:
print(f'{"count":>6}  category')
print('-' * 100)
for cat, n in cat_counter.most_common(50):
    print(f'{n:>6,}  {cat[:90]}')


## 5. Headline diagnosis

In [ ]:
total_ok   = len(expected_ids) - len(zero_benefits)
print(f'Out of {len(expected_ids)} expected benefits:')
print(f'  OK    : {total_ok:>3} extracting data')
print(f'  ZERO  : {n_input_none:>3} because filter found no matching input rows  (DATA SHAPE issue)')
print(f'  ZERO  : {n_llm_failed:>3} despite filter matching rows                  (LLM/PROMPT issue)')

print()
if n_input_none > n_llm_failed:
    print('→ Mostly DATA SHAPE issues. Fix = update BENEFIT_TARGETS keywords/section codes')
    print('  to match how this carrier names categories. See Section 4 above for clues.')
else:
    print('→ Mostly LLM issues. Fix = tighten the per-benefit prompt or check Flask logs.')

# Save the diagnosis for sharing
diag_df = pd.DataFrame(zero_diag)
out_csv = OUTPUT_PATH.stem + '_zero_diagnosis.csv'
diag_df.to_csv(out_csv, index=False)
print(f'\nSaved per-benefit diagnosis to {out_csv}')
